# 04 — Data augmentation et vectorisation texte

Augmentation textuelle (GloVe) sur la colonne layer, puis TextVectorization multi-hot.

In [1]:
import numpy as np
import pandas as pd
from keras import layers

# Fix nlpaug / gensim
from gensim.models import KeyedVectors
import nlpaug.model.word_embs.glove as _glove_module
_glove_module.KeyedVectors = KeyedVectors
import nlpaug.augmenter.word as naw

In [2]:
echantillon_csv = np.load("outputs/echantillon_csv.npy", allow_pickle=True)
deux_dernieres_colonnes = np.load("outputs/deux_dernieres_colonnes.npy", allow_pickle=True)

COL_IDX = 2  # colonne layer
texts = [str(x).strip() for x in echantillon_csv[:, COL_IDX]]

## Data augmentation (GloVe)

In [ ]:
MODEL_PATH = "../../data/glove.6B.300d.txt"
aug_w2v = naw.WordEmbsAug(model_type="glove", model_path=MODEL_PATH, action="substitute", aug_p=0.2)
augmented_texts = aug_w2v.augment(texts, num_thread=8)


In [4]:
# Collecte des lignes augmentées et des indices sources (pour dupliquer les cibles)
augmented_rows = []
idx_sources = []
for idx, aug_text in enumerate(augmented_texts):
    if isinstance(aug_text, list):
        aug_text = aug_text[0] if aug_text else None
    if aug_text is None:
        continue
    aug_str = str(aug_text).strip()
    original_text = str(echantillon_csv[idx, COL_IDX]).strip()
    if aug_str == original_text:
        continue
    new_row = np.array(echantillon_csv[idx].copy(), dtype=object, copy=True)
    new_row[COL_IDX] = aug_str
    augmented_rows.append(new_row)
    idx_sources.append(idx)

if augmented_rows:
    echantillon_csv = np.vstack([echantillon_csv, *augmented_rows])
    targets_aug = deux_dernieres_colonnes[idx_sources]
    deux_dernieres_colonnes = np.vstack([deux_dernieres_colonnes, targets_aug])

print(f"Lignes augmentées : {len(augmented_rows)}. Taille finale : {len(echantillon_csv)} lignes, cibles : {len(deux_dernieres_colonnes)}")

Lignes augmentées : 1304. Taille finale : 3304 lignes, cibles : 3304


## TextVectorization multi-hot (colonne layer)

In [8]:
# Encodage multi-hot de la colonne layer
max_tokens = 20_000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
)
text_vectorization.adapt(echantillon_csv[:, COL_IDX])
col_encoded = np.array(text_vectorization(echantillon_csv[:, COL_IDX]))

# Sauvegarde pour le notebook 05 (assemblage + encodage restant)
np.save("outputs/echantillon_csv_aug.npy", echantillon_csv, allow_pickle=True)
np.save("outputs/col_encoded_multihot.npy", col_encoded)
np.save("outputs/deux_dernieres_colonnes_aug.npy", deux_dernieres_colonnes, allow_pickle=True)
# Sauvegarde manuelle du vocabulaire (corrige problème UnicodeEncodeError de pickle sur l'objet keras)
vocab = text_vectorization.get_vocabulary()
import json
with open("outputs/text_vectorization_vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
# NOTE: Pour restaurer le TextVectorization plus tard : 
# il faudra recharger le vocabulaire puis ré-instancier un layers.TextVectorization avec 'vocabulary=...' 
print(f"Vocabulaire : {len(text_vectorization.get_vocabulary())} tokens. Sauvegardé.")

Vocabulaire : 1628 tokens. Sauvegardé.
